# Sprint 3: GraphSAGE structural-deprivation head

Proposal Sec 3.4.3 (weeks 5–6): train the two-layer GraphSAGE network on Kibera deprivation node labels and hit **macro-F1 ≥ 0.60** on the stratified validation split.

**What this notebook does**

1. Loads (or rebuilds) the Sprint 1 proximity graph from OSM buildings + infrastructure distances.
2. Attaches a three-class UN-Habitat / SUPI-**proxy** deprivation label (`src/utils/deprivation_label.py`) — continuous indicator scores cut into settlement-relative tertiles.
3. Trains `GraphSAGEModel` (Adam lr=`5e-3`, weight decay=`5e-4`, stratified 60/20/20) and reports macro-F1.
4. Saves the best checkpoint for Sprint 5 late fusion.

**About the label.** There is no household survey ground truth in this repo. The class target is an OSM-derived proxy of durability, water/sanitation/road access, and footprint overcrowding. Treat Sprint 3 metrics as *can the GNN learn the proxy*, not as a claim about true SUPI accuracy. Replace the label constructor when field labels exist.

**NRRP note.** Flood-label validation against the NRRP 37 high-risk zones is wired in Sprint 2 v2 (`label_validation.py`) but needs the NRRP polygons (or the OSM river-corridor fallback) run in Colab. Per the proposal, *policy* validation against NRRP is Sprint 5 (fusion engine). Do not block Sprint 3 on it.

## 0. Before you run

- **Runtime → Change runtime type → T4 GPU** (optional; the Kibera graph trains fine on CPU).
- Sprint 1 must have produced `data/graphs/kibera_graph.pt`, **or** this notebook will rebuild the graph live via osmnx (needs Overpass).
- **Checkout the Sprint 3 branch** (`cursor/sprint3-graphsage-deprivation-815d`) — these modules are not on `main` until PR #6 is merged. The clone cell below does this for you.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — GraphSAGE on Kibera is small enough for CPU.")

In [ ]:
%%bash
# Sprint 3 lives on this branch until PR #6 is merged to main.
BRANCH=cursor/sprint3-graphsage-deprivation-815d
cd /content
if [ -d nairobi-risk-mapping ]; then
  cd nairobi-risk-mapping
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git pull origin "$BRANCH"
else
  git clone -b "$BRANCH" https://github.com/emmanuelamor5/nairobi-risk-mapping.git
fi
cd /content/nairobi-risk-mapping && git rev-parse --abbrev-ref HEAD && ls src/utils/deprivation_label.py

In [ ]:
%cd /content/nairobi-risk-mapping
!pip install -q -r requirements.txt

In [ ]:
# --- config knobs ---
SETTLEMENT_KEY = "kibera"
REBUILD_GRAPH = False  # True forces a fresh osmnx pull even if a .pt exists
EPOCHS = 100
LEARNING_RATE = 5e-3
WEIGHT_DECAY = 5e-4
EARLY_STOPPING_PATIENCE = 20
CHECKPOINT_DIR = "checkpoints/sprint3_graphsage"
SEED = 42

## 1. Repo check — Sprint 3 APIs exist

In [ ]:
from src import config
from src.utils.deprivation_label import NODE_FEATURE_NAMES, class_distribution
from src.models.graphsage import GraphSAGE
from src.models.graphsage_model import GraphSAGEModel
from src.models.graphsage_metrics import stratified_node_split

assert len(NODE_FEATURE_NAMES) == 5
assert config.GRAPHSAGE_SPRINT3_MACRO_F1_TARGET == 0.60
print("Node features:", NODE_FEATURE_NAMES)
print("Sprint 3 macro-F1 target:", config.GRAPHSAGE_SPRINT3_MACRO_F1_TARGET)
print("Classes:", config.DEPRIVATION_CLASS_NAMES)

## 2. Load or rebuild the settlement graph

If `data/graphs/{settlement}_graph.pt` was saved by an older Sprint 1 run **without** `data.y` deprivation labels, set `REBUILD_GRAPH = True` so `DataPipeline.acquire_graph_data()` re-enriches buildings and re-serialises.

In [ ]:
from pathlib import Path
import torch
from src.data_pipeline import DataPipeline

graph_path = Path("data") / "graphs" / f"{SETTLEMENT_KEY}_graph.pt"

need_rebuild = REBUILD_GRAPH or (not graph_path.exists())
if not need_rebuild:
    data = torch.load(graph_path, weights_only=False)
    if getattr(data, "y", None) is None or data.x.size(1) != len(NODE_FEATURE_NAMES):
        print("Existing graph lacks deprivation labels or new feature width — rebuilding.")
        need_rebuild = True
    else:
        print(f"Loaded {graph_path}: nodes={data.num_nodes}, edges={data.num_edges}, feats={data.x.size(1)}")

if need_rebuild:
    pipeline = DataPipeline(settlement_key=SETTLEMENT_KEY)
    data, integrity = pipeline.acquire_graph_data()
    print("Graph integrity:", integrity)
    print(f"Rebuilt graph: nodes={data.num_nodes}, edges={data.num_edges}, y={None if data.y is None else data.y.shape}")

In [ ]:
dist = class_distribution(data.y.cpu().numpy())
print("Deprivation class distribution:", dist)

## 3. Train GraphSAGE

In [ ]:
model = GraphSAGEModel(
    in_channels=data.x.size(1),
    hidden_channels=64,
    num_classes=config.DEPRIVATION_NUM_CLASSES,
    dropout=0.5,
)
result = model.fit(
    data,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    checkpoint_dir=CHECKPOINT_DIR,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    seed=SEED,
)
print("Best val macro-F1:", round(result["best_val_macro_f1"], 4))
print("Meets Sprint 3 target (>= 0.60):", result["meets_sprint3_target"])
print("Checkpoint:", result["checkpoint_path"])

In [ ]:
import matplotlib.pyplot as plt

history = result["history"]
epochs = [h["epoch"] for h in history]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(epochs, [h["train_loss"] for h in history], label="train")
ax[0].plot(epochs, [h["val_loss"] for h in history], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(epochs, [h["train_macro_f1"] for h in history], label="train")
ax[1].plot(epochs, [h["val_macro_f1"] for h in history], label="val")
ax[1].axhline(config.GRAPHSAGE_SPRINT3_MACRO_F1_TARGET, color="k", ls="--", label="Sprint 3 bar")
ax[1].set_title("Macro-F1"); ax[1].legend()
plt.tight_layout()
plt.show()

## 4. Hold-out test report

In [ ]:
for split in ("train", "val", "test"):
    report = model.evaluate(data, split=split)
    print(f"\n=== {split} ===")
    print("macro_f1:", round(report["macro_f1"], 4), "accuracy:", round(report["accuracy"], 4))
    print("per_class_f1:", {k: round(v, 4) for k, v in report["per_class_f1"].items()})
    print("support:", report["support"])

## 5. Probability vectors for late fusion (Sprint 5)

The Fusion Engine will concatenate these softmax vectors with the U-Net flood probability per parcel. Persist them alongside the checkpoint.

In [ ]:
probs = model.predict_proba(data)
preds = model.predict(data)
out_path = Path(CHECKPOINT_DIR) / f"{SETTLEMENT_KEY}_deprivation_probs.pt"
torch.save({"probs": probs, "preds": preds, "y": data.y.cpu()}, out_path)
print("Saved", out_path, "shape", tuple(probs.shape))
print("Sample probs[0]:", probs[0].tolist())

## Done

- If `meets_sprint3_target` is True, Sprint 3 exit criterion is met on this run.
- Next up: **Sprint 4** extends both heads to Mathare + Mukuru; **Sprint 5** builds the late-fusion policy matrix and validates against NRRP zones; **Sprint 6** adds SHAP / Grad-CAM + Folium dashboard.
- Reminder: re-run Sprint 2 v2 section 5 in Colab with NRRP polygons (or the OSM river-corridor fallback) when you want quantitative flood-label lift before trusting fusion inputs.